# 02 - 数据库初始化

**功能**: 创建 SQLite 数据库和数据表

In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd

# 数据库路径
PROJECT_ROOT = Path('../../').resolve()
db_path = PROJECT_ROOT / 'data' / 'invest.db'
db_path.parent.mkdir(parents=True, exist_ok=True)

print(f"数据库路径：{db_path}")

# 连接数据库
conn = sqlite3.connect(db_path)
print("✅ 数据库连接成功")

In [ ]:
# 创建数据表
sql_script = """
-- 股票列表
CREATE TABLE IF NOT EXISTS stock_list (
    ts_code TEXT PRIMARY KEY,
    symbol TEXT,
    name TEXT,
    area TEXT,
    industry TEXT,
    market TEXT,
    list_date TEXT
);

-- 日线行情
CREATE TABLE IF NOT EXISTS stock_daily (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    ts_code TEXT NOT NULL,
    trade_date TEXT NOT NULL,
    open REAL, high REAL, low REAL, close REAL,
    pre_close REAL, change REAL, pct_chg REAL,
    vol REAL, amount REAL,
    UNIQUE(ts_code, trade_date)
);
CREATE INDEX IF NOT EXISTS idx_daily_code ON stock_daily(ts_code);
CREATE INDEX IF NOT EXISTS idx_daily_date ON stock_daily(trade_date);

-- 财务指标
CREATE TABLE IF NOT EXISTS stock_fundamentals (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    ts_code TEXT NOT NULL,
    ann_date TEXT NOT NULL,
    end_date TEXT,
    pe REAL, pb REAL, ps REAL,
    roe REAL, roa REAL,
    revenue_growth REAL, profit_growth REAL,
    gross_margin REAL, debt_to_asset REAL,
    UNIQUE(ts_code, ann_date)
);

-- 股东增减持
CREATE TABLE IF NOT EXISTS stock_holder_trade (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    ts_code TEXT NOT NULL,
    holder_name TEXT,
    holder_type TEXT,
    trade_date TEXT,
    trade_type TEXT,
    trade_volume REAL,
    trade_price REAL
);
CREATE INDEX IF NOT EXISTS idx_holder_code ON stock_holder_trade(ts_code);

-- 技术因子计算结果
CREATE TABLE IF NOT EXISTS calc_technical_factors (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    ts_code TEXT NOT NULL,
    trade_date TEXT NOT NULL,
    boll_position REAL,
    boll_bandwidth_pct REAL,
    macd_dif REAL, macd_dif_pct REAL,
    rsi REAL, rsi_pct REAL,
    momentum_20 REAL, momentum_pct REAL,
    volume_ratio REAL, volume_pct REAL,
    volatility REAL, volatility_pct REAL,
    deviation_20 REAL, deviation_pct REAL,
    UNIQUE(ts_code, trade_date)
);
CREATE INDEX IF NOT EXISTS idx_tech_code ON calc_technical_factors(ts_code);

-- 综合置信度
CREATE TABLE IF NOT EXISTS calc_confidence (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    ts_code TEXT NOT NULL,
    calc_date TEXT NOT NULL,
    technical_score REAL,
    fundamental_score REAL,
    event_score REAL,
    confidence REAL,
    signal_type TEXT,
    UNIQUE(ts_code, calc_date)
);

-- 交易信号
CREATE TABLE IF NOT EXISTS signals (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    signal_id TEXT UNIQUE,
    signal_time TEXT NOT NULL,
    ts_code TEXT NOT NULL,
    signal_type TEXT,
    confidence REAL,
    suggested_position REAL,
    stop_loss REAL,
    target_price REAL,
    factors_snapshot TEXT,
    is_pushed INTEGER DEFAULT 0,
    is_executed INTEGER DEFAULT 0
);
CREATE INDEX IF NOT EXISTS idx_signals_code ON signals(ts_code);
"""

# 执行建表
conn.executescript(sql_script)
conn.commit()
print("✅ 数据表创建成功")

In [ ]:
# 验证表创建
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(f"\n数据库中共有 {len(tables)} 张表：")
for table in tables['name']:
    count = pd.read_sql_query(f"SELECT COUNT(*) as count FROM {table}", conn)['count'][0]
    print(f"  - {table}: {count} 条记录")